# Upload a sklearn model to Kaimei

This notebook fits a small logistic regression, uploads the live estimator, and asks Kaimei for one explanation.

```bash
pip install "kaimei[models]" scikit-learn
```

Create an API key in the dashboard, then set `KAIMEI_API_KEY`. For a local API, also set `KAIMEI_BASE_URL=http://localhost:8000`.

In [ ]:
import os
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from kaimei import Kaimei

In [ ]:
bundle = load_breast_cancer()
names = list(bundle.feature_names)
X_train, X_test, y_train, y_test = train_test_split(
    bundle.data, bundle.target, test_size=0.25, random_state=0, stratify=bundle.target
)
clf = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=500)),
])
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

Kaimei stores the extracted JSON spec (scaler + linear weights), not the pickle. Executable structure can be explained without a remote URL.

In [ ]:
client = Kaimei(
    api_key=os.environ["KAIMEI_API_KEY"],
    base_url=os.environ.get("KAIMEI_BASE_URL", "https://api.kaimei.ai"),
)
model = client.models.upload(
    clf,
    name="Breast cancer logistic",
    feature_names=names,
    X=X_train[:400],
    y=y_train[:400],
)
model.id, model.slug

In [ ]:
row = {name: float(value) for name, value in zip(names, X_test[0])}
explanation = client.explanations.create(model=model.slug or model.id, input=row, method="shapley")
explanation.output, explanation.top_features[:5]